In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, precision_recall_curve
from models import ChurnLSTM, ChurnCNN1D, ChurnTransformer

In [2]:
# ==========================================
# 1. 디렉토리 설정
# ==========================================
DATA_DIR = "./data/data/processed"
MODEL_DIR = "./model/deep_learning"
TEST_DIR = "./test"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

print("=" * 50)
print("🚀 [DL] PyTorch 데이터 로드 시작")
print("=" * 50)

🚀 [DL] PyTorch 데이터 로드 시작


In [3]:
# ==========================================
# 2. 미리 분할된 train/valid/test 텐서 및 CSV 로드
# ==========================================
# 텐서 로드
X_train = np.load(os.path.join(DATA_DIR, "train.npy"))
X_val = np.load(os.path.join(DATA_DIR, "valid.npy"))
X_test = np.load(os.path.join(DATA_DIR, "test.npy"))

# CSV 타겟 라벨 로드
train_df = pd.read_csv(os.path.join(DATA_DIR, "train_20181101_v1.csv"))
valid_df = pd.read_csv(os.path.join(DATA_DIR, "valid_20181101_v1.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test_20181101_v1.csv"))

# 타겟 컬럼 자동 감지
def get_target(df):
    target_col = "churn" if "churn" in df.columns else ("is_churn" if "is_churn" in df.columns else df.columns[-1])
    return df[target_col].values

y_train = get_target(train_df)
y_val = get_target(valid_df)
y_test = get_target(test_df)

print(f"Train 텐서 Shape: {X_train.shape}, 타겟 Shape: {y_train.shape}")
print(f"Valid 텐서 Shape: {X_val.shape}, 타겟 Shape: {y_val.shape}")
print(f"Test  텐서 Shape: {X_test.shape}, 타겟 Shape: {y_test.shape}")

Train 텐서 Shape: (12697, 50), 타겟 Shape: (12697,)
Valid 텐서 Shape: (2721, 50), 타겟 Shape: (2721,)
Test  텐서 Shape: (2721, 50), 타겟 Shape: (2721,)


In [4]:
# ==========================================
# 3. DataLoader 구축 (사전 분할된 데이터 사용)
# ==========================================
BATCH_SIZE = 64

train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.long), torch.tensor(y_train, dtype=torch.float32))
val_dataset = TensorDataset(torch.tensor(X_val, dtype=torch.long), torch.tensor(y_val, dtype=torch.float32))
test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.long), torch.tensor(y_test, dtype=torch.float32))

# 클래스 불균형 해결을 위한 Sampler 설정 (Train 데이터 기준)
class_counts = np.bincount(y_train.astype(int))
class_weights = 1.0 / class_counts
sample_weights = class_weights[y_train.astype(int)]
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.double), 
    num_samples=len(sample_weights), 
    replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [5]:
# ==========================================
# 4. Loss 정의
# ==========================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        return torch.mean(self.alpha * (1 - pt)**self.gamma * bce_loss)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = FocalLoss(alpha=0.75, gamma=2.0)
EPOCHS = 20
patience = 4

In [6]:
# ==========================================
# 5. PyTorch LSTM 모델 학습 및 평가
# ==========================================
lstm_model = ChurnLSTM(vocab_size=50, embedding_dim=32, hidden_dim=64, num_layers=1).to(device)
optimizer = optim.Adam(lstm_model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

best_val_loss = float("inf")
patience_counter = 0
best_lstm_path = os.path.join(MODEL_DIR, "best_lstm.pt")

print("\n" + "=" * 50)
print("🚀 [DL] PyTorch LSTM 학습 시작")
print("=" * 50)

for epoch in range(EPOCHS):
    lstm_model.train()
    train_loss = 0
    for seqs, labels in train_loader:
        seqs, labels = seqs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = lstm_model(seqs)
        loss = criterion(outputs.squeeze(), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    lstm_model.eval()
    val_loss = 0
    with torch.no_grad():
        for seqs, labels in val_loader:
            seqs, labels = seqs.to(device), labels.to(device)
            loss = criterion(lstm_model(seqs).squeeze(), labels)
            val_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    scheduler.step(avg_val_loss)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(lstm_model.state_dict(), best_lstm_path)
    else:
        patience_counter += 1
        print(f"  ⚠️ Validation Loss 개선 없음 ({patience_counter}/{patience})")
        if patience_counter >= patience:
            print(f"🛑 LSTM 조기 종료 발동! (Epoch {epoch+1})")
            lstm_model.load_state_dict(torch.load(best_lstm_path))
            break

# LSTM 테스트셋 예측 및 저장
lstm_model.eval()
lstm_preds_list = []
with torch.no_grad():
    for seqs, _ in test_loader:
        seqs = seqs.to(device)
        outputs = torch.sigmoid(lstm_model(seqs))
        lstm_preds_list.extend(outputs.cpu().numpy())

lstm_preds_proba = np.array(lstm_preds_list).ravel()
lstm_auc = roc_auc_score(y_test, lstm_preds_proba)
print(f"\n🚀 [LSTM 단독] Test ROC-AUC Score: {lstm_auc:.4f}")

torch.save(lstm_model.state_dict(), os.path.join(MODEL_DIR, "lstm_model_weights.pth"))
np.save(os.path.join(TEST_DIR, "lstm_preds_proba.npy"), lstm_preds_proba)
np.save(os.path.join(TEST_DIR, "test_y_true.npy"), y_test)
print("✅ LSTM 모델 및 예측값 저장 완료!")


🚀 [DL] PyTorch LSTM 학습 시작
Epoch [1/20] | Train Loss: 0.0887 | Val Loss: 0.0919
Epoch [2/20] | Train Loss: 0.0806 | Val Loss: 0.0866
Epoch [3/20] | Train Loss: 0.0811 | Val Loss: 0.0841
Epoch [4/20] | Train Loss: 0.0787 | Val Loss: 0.0774
Epoch [5/20] | Train Loss: 0.0769 | Val Loss: 0.0787
  ⚠️ Validation Loss 개선 없음 (1/4)
Epoch [6/20] | Train Loss: 0.0745 | Val Loss: 0.0731
Epoch [7/20] | Train Loss: 0.0747 | Val Loss: 0.0742
  ⚠️ Validation Loss 개선 없음 (1/4)
Epoch [8/20] | Train Loss: 0.0738 | Val Loss: 0.0895
  ⚠️ Validation Loss 개선 없음 (2/4)
Epoch [9/20] | Train Loss: 0.0734 | Val Loss: 0.0762
  ⚠️ Validation Loss 개선 없음 (3/4)
Epoch [10/20] | Train Loss: 0.0741 | Val Loss: 0.0792
  ⚠️ Validation Loss 개선 없음 (4/4)
🛑 LSTM 조기 종료 발동! (Epoch 10)

🚀 [LSTM 단독] Test ROC-AUC Score: 0.8685
✅ LSTM 모델 및 예측값 저장 완료!


In [7]:
# ==========================================
# 6. PyTorch CNN1D 모델 학습 및 평가 (추가됨)
# ==========================================
cnn_model = ChurnCNN1D(vocab_size=50, embedding_dim=32, num_filters=64).to(device)
optimizer = optim.Adam(cnn_model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

best_val_loss = float("inf")
patience_counter = 0
best_cnn_path = os.path.join(MODEL_DIR, "best_cnn.pt")

print("\n" + "=" * 50)
print("🚀 [DL] PyTorch CNN1D 학습 시작")
print("=" * 50)

for epoch in range(EPOCHS):
    cnn_model.train()
    train_loss = 0
    for seqs, labels in train_loader:
        seqs, labels = seqs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = cnn_model(seqs)
        loss = criterion(outputs.squeeze(), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    cnn_model.eval()
    val_loss = 0
    with torch.no_grad():
        for seqs, labels in val_loader:
            seqs, labels = seqs.to(device), labels.to(device)
            loss = criterion(cnn_model(seqs).squeeze(), labels)
            val_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    scheduler.step(avg_val_loss)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(cnn_model.state_dict(), best_cnn_path)
    else:
        patience_counter += 1
        print(f"  ⚠️ Validation Loss 개선 없음 ({patience_counter}/{patience})")
        if patience_counter >= patience:
            print(f"🛑 CNN1D 조기 종료 발동! (Epoch {epoch+1})")
            cnn_model.load_state_dict(torch.load(best_cnn_path))
            break

# CNN1D 테스트셋 예측 및 저장
cnn_model.eval()
cnn_preds_list = []
with torch.no_grad():
    for seqs, _ in test_loader:
        seqs = seqs.to(device)
        outputs = torch.sigmoid(cnn_model(seqs))
        cnn_preds_list.extend(outputs.cpu().numpy())

cnn_preds_proba = np.array(cnn_preds_list).ravel()
cnn_auc = roc_auc_score(y_test, cnn_preds_proba)
print(f"\n🚀 [CNN1D 단독] Test ROC-AUC Score: {cnn_auc:.4f}")

torch.save(cnn_model.state_dict(), os.path.join(MODEL_DIR, "cnn1d_model_weights.pth"))
np.save(os.path.join(TEST_DIR, "cnn_preds_proba.npy"), cnn_preds_proba)
print("✅ CNN1D 모델 및 예측값 저장 완료!")


🚀 [DL] PyTorch CNN1D 학습 시작
Epoch [1/20] | Train Loss: 0.0979 | Val Loss: 0.0833
Epoch [2/20] | Train Loss: 0.0762 | Val Loss: 0.0803
Epoch [3/20] | Train Loss: 0.0731 | Val Loss: 0.0769
Epoch [4/20] | Train Loss: 0.0712 | Val Loss: 0.0826
  ⚠️ Validation Loss 개선 없음 (1/4)
Epoch [5/20] | Train Loss: 0.0684 | Val Loss: 0.0809
  ⚠️ Validation Loss 개선 없음 (2/4)
Epoch [6/20] | Train Loss: 0.0663 | Val Loss: 0.0793
  ⚠️ Validation Loss 개선 없음 (3/4)
Epoch [7/20] | Train Loss: 0.0631 | Val Loss: 0.0748
Epoch [8/20] | Train Loss: 0.0612 | Val Loss: 0.0719
Epoch [9/20] | Train Loss: 0.0600 | Val Loss: 0.0736
  ⚠️ Validation Loss 개선 없음 (1/4)
Epoch [10/20] | Train Loss: 0.0574 | Val Loss: 0.0819
  ⚠️ Validation Loss 개선 없음 (2/4)
Epoch [11/20] | Train Loss: 0.0575 | Val Loss: 0.0763
  ⚠️ Validation Loss 개선 없음 (3/4)
Epoch [12/20] | Train Loss: 0.0560 | Val Loss: 0.0751
  ⚠️ Validation Loss 개선 없음 (4/4)
🛑 CNN1D 조기 종료 발동! (Epoch 12)

🚀 [CNN1D 단독] Test ROC-AUC Score: 0.8453
✅ CNN1D 모델 및 예측값 저장 완료!


In [8]:
# ==========================================
# 7. PyTorch Transformer 모델 학습 및 평가
# ==========================================
transformer_model = ChurnTransformer(
    vocab_size=50, embedding_dim=32, num_heads=4, hidden_dim=64, num_layers=2, dropout=0.1
).to(device)

optimizer = optim.Adam(transformer_model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

best_val_loss = float("inf")
patience_counter = 0
best_transformer_path = os.path.join(MODEL_DIR, "best_transformer.pt")

print("\n" + "=" * 50)
print("🚀 [DL] PyTorch Transformer 학습 시작")
print("=" * 50)

for epoch in range(EPOCHS):
    transformer_model.train()
    train_loss = 0
    for seqs, labels in train_loader:
        seqs, labels = seqs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = transformer_model(seqs)
        loss = criterion(outputs.squeeze(), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    transformer_model.eval()
    val_loss = 0
    with torch.no_grad():
        for seqs, labels in val_loader:
            seqs, labels = seqs.to(device), labels.to(device)
            loss = criterion(transformer_model(seqs).squeeze(), labels)
            val_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    scheduler.step(avg_val_loss)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(transformer_model.state_dict(), best_transformer_path)
    else:
        patience_counter += 1
        print(f"  ⚠️ Validation Loss 개선 없음 ({patience_counter}/{patience})")
        if patience_counter >= patience:
            print(f"🛑 Transformer 조기 종료 발동! (Epoch {epoch+1})")
            transformer_model.load_state_dict(torch.load(best_transformer_path))
            break

# Transformer 테스트셋 예측 및 저장
transformer_model.eval()
transformer_preds_list = []
with torch.no_grad():
    for seqs, _ in test_loader:
        seqs = seqs.to(device)
        outputs = torch.sigmoid(transformer_model(seqs))
        transformer_preds_list.extend(outputs.cpu().numpy())

transformer_preds_proba = np.array(transformer_preds_list).ravel()
transformer_auc = roc_auc_score(y_test, transformer_preds_proba)
print(f"\n🚀 [Transformer 단독] Test ROC-AUC Score: {transformer_auc:.4f}")

torch.save(transformer_model.state_dict(), os.path.join(MODEL_DIR, "transformer_model_weights.pth"))
np.save(os.path.join(TEST_DIR, "transformer_preds_proba.npy"), transformer_preds_proba)
print("✅ Transformer 모델 및 예측값 저장 완료!")


🚀 [DL] PyTorch Transformer 학습 시작
Epoch [1/20] | Train Loss: 0.1071 | Val Loss: 0.0738
Epoch [2/20] | Train Loss: 0.0775 | Val Loss: 0.0765
  ⚠️ Validation Loss 개선 없음 (1/4)
Epoch [3/20] | Train Loss: 0.0742 | Val Loss: 0.0976
  ⚠️ Validation Loss 개선 없음 (2/4)
Epoch [4/20] | Train Loss: 0.0723 | Val Loss: 0.0920
  ⚠️ Validation Loss 개선 없음 (3/4)
Epoch [5/20] | Train Loss: 0.0701 | Val Loss: 0.0634
Epoch [6/20] | Train Loss: 0.0694 | Val Loss: 0.0867
  ⚠️ Validation Loss 개선 없음 (1/4)
Epoch [7/20] | Train Loss: 0.0676 | Val Loss: 0.0730
  ⚠️ Validation Loss 개선 없음 (2/4)
Epoch [8/20] | Train Loss: 0.0676 | Val Loss: 0.0749
  ⚠️ Validation Loss 개선 없음 (3/4)
Epoch [9/20] | Train Loss: 0.0662 | Val Loss: 0.0705
  ⚠️ Validation Loss 개선 없음 (4/4)
🛑 Transformer 조기 종료 발동! (Epoch 9)

🚀 [Transformer 단독] Test ROC-AUC Score: 0.8631
✅ Transformer 모델 및 예측값 저장 완료!
